# Grounded QA Demo

This notebook exercises the local baseline: evidence chunks are retrieved in memory, an extractive cited sentence is selected, and the grounding verifier accepts or rejects the answer.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "libs/schema"), str(ROOT)]

from sciqa_schema import EvidenceChunk, GeneratedSentence
from services.query.app.core.grounding import GroundingVerifier
from services.query.app.core.local_rag import answer_question_local

## Answerable Question

The local baseline retrieves the best chunk, extracts the cited sentence, and returns a verified sentence with supporting chunk IDs.

In [ ]:
chunks = [
    EvidenceChunk(
        chunk_id="paper-alpha:passage:0",
        doc_id="paper-alpha",
        text=(
            "The retrieval-augmented model improved F1 by 4.2 points on SciFact. "
            "The baseline used sparse retrieval only."
        ),
    )
]

result = answer_question_local(
    "How much did the retrieval-augmented model improve F1?",
    chunks,
)
result.model_dump(mode="json")

## Abstention Case

When no retrieved evidence overlaps the question, the local baseline abstains instead of fabricating an answer.

In [ ]:
abstention = answer_question_local(
    "Which hidden calibration seed was used?",
    [
        EvidenceChunk(
            chunk_id="paper-delta:passage:0",
            doc_id="paper-delta",
            text="The paper reported accuracy and F1 for the classifier.",
        )
    ],
)
abstention.model_dump(mode="json")

## Numeric Claim Rejection

The verifier rejects a generated numeric claim when the number does not appear in the cited evidence.

In [ ]:
verifier = GroundingVerifier()
decision = verifier.verify(
    [
        GeneratedSentence(
            text="The method improved F1 by 9.8 points.",
            supporting_chunk_ids=["paper-alpha:passage:0"],
        )
    ],
    chunks,
)
decision.model_dump(mode="json")